# QA-OCR_LLM: OCR Pipeline Experiment Evaluation

**Approach**: OCR (Azure/Mistral) → Text Extraction → LLM QA

This notebook evaluates the OCR-based pipeline for document QA:
- **QA-OCR_LLM_simple**: Simple prompt
- **QA-OCR_LLM_detailed**: Detailed prompt  
- **QA-OCR_LLM_cot**: Chain-of-Thought prompt

**Datasets**: DocVQA_mini, InfographicVQA_mini

**Metrics**: ANLS (primary), Exact Match

In [5]:
# Setup and imports
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, Markdown

# Import shared utilities
from ocr_vs_vlm.results_final.shared.colors import (
    MODEL_COLORS, DATASET_COLORS, get_model_color, get_dataset_color
)
from ocr_vs_vlm.results_final.shared.stats_utils import (
    bootstrap_ci, wilcoxon_test, cohens_d, compute_significance_matrix, format_pvalue
)
from ocr_vs_vlm.results_final.shared.viz_utils import (
    setup_plotly_template, create_metric_boxplot, create_heatmap,
    create_grouped_bar_chart, create_significance_heatmap
)
from ocr_vs_vlm.results_final.shared.data_loader import (
    load_experiment_data, load_dataset_data, extract_models_from_columns,
    extract_metric_scores, compute_summary_statistics
)

# Setup Plotly template
setup_plotly_template()
print("✓ Setup complete")

✓ Setup complete


## 1. Load QA1 Experiment Data

In [6]:
# Load QA1 phases for both datasets
QA1_PHASES = ['QA-OCR_LLM_simple', 'QA-OCR_LLM_detailed', 'QA-OCR_LLM_cot']
DATASETS = ['DocVQA_mini', 'InfographicVQA_mini']

# Load and combine data
all_data = []
for dataset in DATASETS:
    for phase in QA1_PHASES:
        try:
            df = load_experiment_data(dataset, phase)
            all_data.append(df)
            print(f"✓ Loaded {dataset}/{phase}: {len(df)} samples")
        except FileNotFoundError as e:
            print(f"✗ Missing {dataset}/{phase}")

df_qa1 = pd.concat(all_data, ignore_index=True)
print(f"\n📊 Total samples: {len(df_qa1)}")

# Extract available models
models = extract_models_from_columns(df_qa1)
print(f"📋 Models: {models}")

✗ Missing DocVQA_mini/QA-OCR_LLM_simple
✗ Missing DocVQA_mini/QA-OCR_LLM_detailed
✗ Missing DocVQA_mini/QA-OCR_LLM_cot
✗ Missing InfographicVQA_mini/QA-OCR_LLM_simple
✗ Missing InfographicVQA_mini/QA-OCR_LLM_detailed
✗ Missing InfographicVQA_mini/QA-OCR_LLM_cot


ValueError: No objects to concatenate

## 2. OCR Model Comparison (Azure vs Mistral)

In [ ]:
# Extract ANLS scores by model
anls_scores = extract_metric_scores(df_qa1, 'anls_score')

# Compute summary statistics with bootstrap CI
summary_rows = []
for model, scores in anls_scores.items():
    scores_clean = scores[~np.isnan(scores)]
    mean, ci_lo, ci_hi = bootstrap_ci(scores_clean, 'mean')
    summary_rows.append({
        'Model': model,
        'ANLS Mean': mean,
        'ANLS CI (95%)': f"[{ci_lo:.4f}, {ci_hi:.4f}]",
        'N': len(scores_clean)
    })

summary_df = pd.DataFrame(summary_rows).sort_values('ANLS Mean', ascending=False)
display(summary_df)

# Create boxplot
fig = create_metric_boxplot(anls_scores, metric_name='ANLS Score', 
                            title='QA1 (OCR Pipeline): ANLS by OCR Model')
fig.show()

In [ ]:
# Statistical significance test (Wilcoxon)
if len(anls_scores) >= 2:
    model_names = list(anls_scores.keys())
    p_values, significance = compute_significance_matrix(anls_scores, test='wilcoxon')
    
    print("📊 Pairwise Wilcoxon Signed-Rank Tests (Bonferroni corrected)")
    print("-" * 60)
    for (a, b), p in p_values.items():
        d = cohens_d(anls_scores[a], anls_scores[b])
        sig = "✓ Significant" if significance[(a, b)] else "✗ Not significant"
        print(f"{a} vs {b}: p={format_pvalue(p)}, Cohen's d={d:.3f} ({sig})")

## 3. Prompt Effect Analysis (Simple vs Detailed vs CoT)

In [ ]:
# Analyze by prompt type
PROMPT_LABELS = {'QA-OCR_LLM_simple': 'Simple', 'QA-OCR_LLM_detailed': 'Detailed', 'QA-OCR_LLM_cot': 'Chain-of-Thought'}

# Aggregate by phase (prompt type)
phase_stats = []
for phase in QA1_PHASES:
    phase_df = df_qa1[df_qa1['phase'] == phase]
    for model in models:
        col = f'anls_score_{model}'
        if col in phase_df.columns:
            scores = phase_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                phase_stats.append({
                    'Phase': phase,
                    'Prompt': PROMPT_LABELS[phase],
                    'Model': model,
                    'ANLS': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi,
                    'Error': mean - ci_lo  # For error bars
                })

phase_df = pd.DataFrame(phase_stats)
display(phase_df.pivot_table(index='Prompt', columns='Model', values='ANLS', aggfunc='mean').round(4))

In [ ]:
# Grouped bar chart: Prompt effect by model
fig = create_grouped_bar_chart(
    phase_df, 
    x_col='Prompt', 
    y_col='ANLS', 
    group_col='Model',
    title='QA1: ANLS by Prompt Type',
    y_label='ANLS Score',
    error_col='Error'
)
fig.update_layout(xaxis_categoryorder='array', xaxis_categoryarray=['Simple', 'Detailed', 'Chain-of-Thought'])
fig.show()

## 4. Dataset Comparison (DocVQA vs InfographicVQA)

In [ ]:
# Analyze by dataset
dataset_stats = []
for dataset in DATASETS:
    dataset_df = df_qa1[df_qa1['dataset'] == dataset]
    for model in models:
        col = f'anls_score_{model}'
        if col in dataset_df.columns:
            scores = dataset_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                dataset_stats.append({
                    'Dataset': dataset.replace('_mini', ''),
                    'Model': model,
                    'ANLS': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi
                })

dataset_summary = pd.DataFrame(dataset_stats)

# Create comparison chart
fig = go.Figure()
for dataset in dataset_summary['Dataset'].unique():
    ds_data = dataset_summary[dataset_summary['Dataset'] == dataset]
    fig.add_trace(go.Bar(
        x=ds_data['Model'],
        y=ds_data['ANLS'],
        name=dataset,
        marker_color=get_dataset_color(dataset + '_mini'),
        error_y=dict(type='data', array=ds_data['ANLS'] - ds_data['CI_Lower'], visible=True)
    ))

fig.update_layout(
    title='QA1: ANLS by Dataset',
    yaxis_title='ANLS Score',
    barmode='group',
    height=450
)
fig.show()

## 5. Summary & Key Findings

In [ ]:
# Generate summary report
print("=" * 70)
print("QA1 (OCR PIPELINE) EXPERIMENT SUMMARY")
print("=" * 70)

# Best performing configuration
if len(phase_df) > 0:
    best = phase_df.loc[phase_df['ANLS'].idxmax()]
    print(f"\n🏆 Best Configuration:")
    print(f"   Model: {best['Model']}")
    print(f"   Prompt: {best['Prompt']}")
    print(f"   ANLS: {best['ANLS']:.4f} [{best['CI_Lower']:.4f}, {best['CI_Upper']:.4f}]")

# Overall model ranking
print(f"\n📊 Model Ranking (Overall ANLS):")
for i, row in summary_df.iterrows():
    print(f"   {row['Model']}: {row['ANLS Mean']:.4f} {row['ANLS CI (95%)']}")

# Prompt effect
print(f"\n💡 Prompt Effect:")
prompt_means = phase_df.groupby('Prompt')['ANLS'].mean().sort_values(ascending=False)
for prompt, anls in prompt_means.items():
    print(f"   {prompt}: {anls:.4f}")

print("\n" + "=" * 70)